# 30 · Segments of the exposed supply

Inputs: `holder_snapshot`, `utxo_life`, `address_key`, `address_stats`, `key_exposure`, `exchange_keys`, `custodial_keys` (02, 04, 04b, 04c)

Outputs: `q2_segments.csv`, `q2_key_number.csv`, `q2_sensitivity.csv`, `q2_hetero_bal.csv`, `q2_hetero_fsy.csv`, `q2_coin_decomp.csv`, `q2_bridge.csv`, `q2_cells.csv.gz`, `fig_30_segments.png`, `fig_30_sensitivity.png`, `fig_30_bridge.png`, `build_report_30.json`

Coins held by each group-L (and T) key on the day before Willow (2024-12-08, "starting coins") are followed UTXO by UTXO to the freeze date (2026-09-22). Each key is assigned to one class, in order:

| Class | Rule (key level) |
|---|---|
| Exchange/custodian | exchange label, Coinbase migration key, institution identified in 04c, or any address with > 1,000 receipts (as in the event study) |
| A Confirmed migrated | starting coins spent, balance at the freeze date < 0.01 BTC, more than half of the outflow to y1 strict |
| B Moved, still exposed | starting coins spent, balance at the freeze date < 0.01 BTC, main destination y0, an exchange or unclear |
| C Active, not migrated | balance at the freeze date ≥ 0.01 BTC and a spend within 3 years before the freeze date |
| D Indeterminate | balance at the freeze date ≥ 0.01 BTC, last spend 3–10 years before (never-spent keys: last major receipt within 10 years) |
| E Behaviorally unreachable | balance at the freeze date ≥ 0.01 BTC, last spend (never-spent keys: last major receipt) more than 10 years before; a lower bound, not an estimate of lost coins; Patoshi flagged separately |
| Dust | starting balance < 0.01 BTC and starting coins unmoved |

- BTC are counted at the starting balance, so classes sum to the starting exposed supply; the freeze-date exposed balance of class C is also reported.
- Major receipt: a single receipt ≥ 1% of the starting balance. Used for never-spent keys so that dust sent to old keys (about 26,000 never-spent keys with about 1.3 million BTC, mostly Patoshi, received dust around 2020–21) does not count as activity.

In [ ]:
# Configuration
PROJECT_ID = "your-gcp-project"
DATASET    = "qbtc_raw"
LOCATION   = "US"

AUTO_RUN_USD      = 5.0
MAX_USD_PER_QUERY = 10.0
MAX_USD_TOTAL     = 100.0
PRICE_PER_TIB_USD = 6.25
CONFIRM           = False
FORCE_REBUILD     = False

START_EVENT      = "willow"      # starting snapshot: the day before Willow, 2024-12-08
FREEZE_DATE      = "2026-09-22"  # freeze date = latest snapshot date
MIN_BTC          = 0.01          # threshold for "still holds an exposed balance"
SERVICE_MAX_RECV = 1000          # any address with > 1,000 lifetime receipts -> service/custodian (as in 04)
DORMANT_DAYS     = 30            # y1 strict: unspent for the next 30 days (as in 04)
Y1S_MAJORITY     = 0.5           # class A: threshold on the y1-strict share of the outflow
ACTIVE_YEARS     = 3             # main definition: spent within 3 years before the freeze date = active
DORMANT_YEARS    = 10            # main definition: last spend >= 10 years ago = behaviorally unreachable
SENS_ACTIVE      = [1, 3, 5]
SENS_DORMANT     = [5, 10, 15]
MAJOR_IN_SHARE   = 0.01          # for never-spent keys, dormancy uses the last receipt >= 1% of the starting balance (dust ignored)
EARLY_END        = "2010-12-31"  # early-mining flag: all starting UTXOs are P2PK created before this day
PATOSHI_URL = "https://raw.githubusercontent.com/bensig/patoshi-addresses/main/patoshi_pubkeys_COMPLETE.csv"

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

from google.cloud import bigquery
import json, os, io, re, time, datetime as dt
import numpy as np
import pandas as pd
import requests

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DS  = f"{PROJECT_ID}.{DATASET}"
PUB = "bigquery-public-data.crypto_bitcoin"

LEDGER_PATH, JOBS_PATH = "build_ledger.json", "jobs.json"
LEDGER = json.load(open(LEDGER_PATH)) if os.path.exists(LEDGER_PATH) else {"jobs": [], "usd_total": 0.0}
JOBS   = json.load(open(JOBS_PATH))   if os.path.exists(JOBS_PATH)   else []

for t in ["utxo_life", "address_key", "address_stats", "key_exposure", "holder_snapshot",
          "exchange_keys", "custodial_keys"]:
    client.get_table(f"{DS}.{t}")        # raises if a table is missing
print("Tables from earlier steps found. Spent in this Colab session: $%.2f / $%.0f" % (LEDGER["usd_total"], MAX_USD_TOTAL))

In [ ]:
# Helpers
def usd(gb): return gb / 1024 * PRICE_PER_TIB_USD

def dry_run(sql):
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = job.total_bytes_processed / 2**30
    return gb, usd(gb)

def table_rows(name):
    try:
        return client.get_table(f"{DS}.{name}").num_rows
    except Exception:
        return None

def table_info(name):
    t = client.get_table(f"{DS}.{name}")
    return {"table": name, "rows": t.num_rows, "GB": round(t.num_bytes / 2**30, 2)}

def build(name, sql, force=False):
    assert not re.search(r"CLUSTER BY[^\n]*\b(key_id|txid|spending_txid)\b", sql), f"[{name}] cannot cluster on a BYTES column"
    n = table_rows(name)
    if n and not (FORCE_REBUILD or force):
        print(f"[{name}] exists, {n:,} rows; skipped (rebuild with build(..., force=True))")
        return True
    gb, cost = dry_run(sql)
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{name}] estimated ${cost:.2f}, above the per-query limit ${MAX_USD_PER_QUERY}")
    if LEDGER["usd_total"] + cost > MAX_USD_TOTAL:
        raise RuntimeError(f"[{name}] total would exceed ${MAX_USD_TOTAL}")
    if cost > AUTO_RUN_USD and not CONFIRM:
        raise RuntimeError(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}, above the auto-run limit ${AUTO_RUN_USD}: set CONFIRM=True and re-run this cell (tables already built are skipped)")
    print(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}")
    job = client.query(sql, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30, use_query_cache=False))
    JOBS.append({"table": name, "job_id": job.job_id, "location": LOCATION,
                 "submitted_at": dt.datetime.now(dt.timezone.utc).isoformat()})
    json.dump(JOBS, open(JOBS_PATH, "w"), indent=2)
    print(f"[{name}] submitted job {job.job_id} (runs on the server; finishes even if Colab disconnects)")
    t0 = time.time()
    job.result()
    billed = (job.total_bytes_billed or 0) / 2**30
    LEDGER["jobs"].append({"table": name, "job_id": job.job_id, "billed_gb": round(billed, 1),
                           "usd": round(usd(billed), 2), "seconds": round(time.time() - t0, 1)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + usd(billed), 2)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    print(f"[{name}] done, {billed:,.0f} GB, {LEDGER['jobs'][-1]['seconds']}s", table_info(name))
    return True

def need(*names):
    missing = [n for n in names if not table_rows(n)]
    assert not missing, f"These tables are not built yet: {missing} (run the earlier cells first)"

def q(sql):
    return client.query(sql).to_dataframe()

def upload(name, df):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.date
    job = client.load_table_from_dataframe(
        df, f"{DS}.{name}", job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    job.result()
    print(f"[{name}] uploaded {len(df):,} rows")

In [ ]:
# Dates and constants
SATS_MIN = int(round(MIN_BTC * 1e8))
FREEZE = pd.Timestamp(FREEZE_DATE)
snap = q(f"""SELECT event, ANY_VALUE(snap_date) AS snap_date FROM `{DS}.holder_snapshot`
             WHERE event IN ('{START_EVENT}', 'latest') GROUP BY event""").set_index("event").snap_date
START = pd.Timestamp(snap[START_EVENT])
assert pd.Timestamp(snap["latest"]) == FREEZE, f"latest snapshot date {snap['latest']} differs from the freeze date {FREEZE_DATE}"
S, S1, F, F1 = (START.strftime("%Y-%m-%d"), (START + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
                FREEZE.strftime("%Y-%m-%d"), (FREEZE + pd.Timedelta(days=1)).strftime("%Y-%m-%d"))
STD_TYPES  = "('pubkey','pubkeyhash','witness_v0_keyhash','scripthash','witness_v0_scripthash','witness_v1_taproot')"
HASH_TYPES = "('pubkeyhash','witness_v0_keyhash','scripthash','witness_v0_scripthash')"
print(f"Starting snapshot {S} (spends tracked from {S1}), freeze date {F}")

## Step 1 · Patoshi block list and starting keys, `q2_universe`

Starting keys: group L and T keys in the willow snapshot of `holder_snapshot` (2024-12-08) with a positive balance.

Patoshi blocks from `bensig/patoshi-addresses` (pattern identified by Sergio Lerner, block list compiled by Jameson Lopp; 21,953 blocks, heights 3–49,973), uploaded as `patoshi_blocks`. A key with a starting P2PK UTXO in one of these blocks is flagged. If the download fails, only the early-mining flag (2009–2010, all P2PK, never spent) is used.

In [ ]:
try:
    if os.path.exists("patoshi_pubkeys_COMPLETE.csv"):
        pat_raw = pd.read_csv("patoshi_pubkeys_COMPLETE.csv")
    else:
        pat_raw = pd.read_csv(io.StringIO(requests.get(PATOSHI_URL, timeout=120).text))
    col = [c for c in pat_raw.columns if "height" in c.lower()][0]
    patoshi = pd.DataFrame({"height": sorted(set(pat_raw[col].astype(int)))})
    PATOSHI_SRC = PATOSHI_URL
    print(f"Patoshi blocks {len(patoshi):,}, heights {patoshi.height.min()}-{patoshi.height.max()}, "
          f"list total {pat_raw.iloc[:, 3].sum():,.0f} BTC")
    assert 20000 <= len(patoshi) <= 23000, "number of blocks far from the known 21,953; check the file"
except Exception as e:
    print("Patoshi list download/parsing failed; using the early-mining flag only:", e)
    patoshi = pd.DataFrame({"height": pd.Series([], dtype="int64")})
    PATOSHI_SRC = None
upload("patoshi_blocks", patoshi if len(patoshi) else pd.DataFrame({"height": [-1]}))

SQL_UNIV = f"""
CREATE OR REPLACE TABLE `{DS}.q2_universe` AS
SELECT key_id, grp, key_kind, sats AS s0_sats, n_utxo AS s0_n_utxo,
       oldest_utxo_date, newest_utxo_date, first_exposed_date, first_exposed_src
FROM `{DS}.holder_snapshot`
WHERE event = '{START_EVENT}' AND grp IN ('L', 'T') AND sats > 0
"""
build("q2_universe", SQL_UNIV)
display(q(f"""SELECT grp, COUNT(*) AS n_keys, ROUND(SUM(s0_sats)/1e8) AS btc
              FROM `{DS}.q2_universe` GROUP BY grp ORDER BY grp"""))

## Step 2 · Key attributes, `q2_key_attr`

Per starting key: first-seen date, last spend, last receipt, maximum receipts at one address (extremes over the key's addresses in `address_stats`); balance at the freeze date (latest snapshot, 0 if absent); exchange/custodian/service flag.

In [ ]:
need("q2_universe")
SQL_ATTR = f"""
CREATE OR REPLACE TABLE `{DS}.q2_key_attr` AS
WITH ka AS (
  SELECT k.key_id,
         MIN(s.first_seen_date)    AS first_seen_date,
         MAX(s.last_spend_date)    AS last_spend_date,
         MAX(s.last_received_date) AS last_received_date,
         MAX(s.n_received)         AS max_addr_n_received,
         COUNT(*)                  AS n_addr_total
  FROM `{DS}.address_key` k
  JOIN `{DS}.q2_universe` u ON u.key_id = k.key_id
  JOIN `{DS}.address_stats` s ON s.address = k.address
  GROUP BY k.key_id
),
lat AS (SELECT key_id, sats AS end_sats FROM `{DS}.holder_snapshot` WHERE event = 'latest'),
ck  AS (SELECT DISTINCT key_id FROM `{DS}.custodial_keys`)
SELECT ka.key_id, ka.first_seen_date, ka.last_spend_date, ka.last_received_date,
       ka.max_addr_n_received, ka.n_addr_total,
       IFNULL(lat.end_sats, 0)  AS end_sats,
       ex.key_id IS NOT NULL    AS is_exchange,
       ck.key_id IS NOT NULL    AS is_custodial,
       (ex.key_id IS NOT NULL OR ck.key_id IS NOT NULL
        OR IFNULL(ka.max_addr_n_received, 0) > {SERVICE_MAX_RECV}) AS is_service
FROM ka
LEFT JOIN lat ON lat.key_id = ka.key_id
LEFT JOIN `{DS}.exchange_keys` ex ON ex.key_id = ka.key_id
LEFT JOIN ck ON ck.key_id = ka.key_id
"""
build("q2_key_attr", SQL_ATTR)
display(q(f"""SELECT u.grp, COUNT(*) AS n_keys, COUNTIF(a.key_id IS NULL) AS n_missing_attr,
                     COUNTIF(a.is_exchange) AS n_exchange, COUNTIF(a.is_custodial) AS n_custodial,
                     COUNTIF(a.is_service) AS n_service,
                     ROUND(SUM(IF(a.is_service, u.s0_sats, 0))/1e8) AS service_btc
              FROM `{DS}.q2_universe` u LEFT JOIN `{DS}.q2_key_attr` a ON a.key_id = u.key_id
              GROUP BY u.grp ORDER BY u.grp"""))

## Step 3 · Starting coins, `q2_start_utxo`

UTXOs alive at the close of 2024-12-08 (six standard script types) belonging to starting keys; spends after the freeze date are treated as unspent. Check: UTXO values sum to each key's starting balance.

In [ ]:
need("q2_universe", "patoshi_blocks")
SQL_START = f"""
CREATE OR REPLACE TABLE `{DS}.q2_start_utxo` AS
WITH pb AS (SELECT DISTINCT height FROM `{DS}.patoshi_blocks`)
SELECT k.key_id, un.grp, u.txid, u.vout, u.sats, u.type, u.created_date, u.created_height,
       IF(u.spent_date <= DATE '{F}', u.spent_date, NULL)    AS spent_date,
       IF(u.spent_date <= DATE '{F}', u.spending_txid, NULL) AS spending_txid,
       (u.type = 'pubkey' AND pb.height IS NOT NULL)         AS is_patoshi
FROM `{DS}.utxo_life` u
JOIN `{DS}.address_key` k ON k.address = u.address
JOIN `{DS}.q2_universe` un ON un.key_id = k.key_id
LEFT JOIN pb ON pb.height = u.created_height
WHERE u.created_date <= DATE '{S}'
  AND (u.spent_date IS NULL OR u.spent_date > DATE '{S}')
  AND u.type IN {STD_TYPES}
  AND u.address IS NOT NULL
"""
build("q2_start_utxo", SQL_START)
chk_start = q(f"""
WITH s AS (SELECT key_id, SUM(sats) AS s, COUNT(*) AS n FROM `{DS}.q2_start_utxo` GROUP BY key_id)
SELECT COUNT(*) AS n_keys, COUNTIF(s.key_id IS NULL) AS keys_without_utxo,
       COUNTIF(s.s != u.s0_sats) AS keys_sats_mismatch, COUNTIF(s.n != u.s0_n_utxo) AS keys_n_mismatch,
       ROUND(SUM(u.s0_sats)/1e8) AS btc_snapshot, ROUND(SUM(s.s)/1e8) AS btc_utxo
FROM `{DS}.q2_universe` u LEFT JOIN s ON s.key_id = u.key_id""")
display(chk_start)
assert chk_start.keys_without_utxo.iloc[0] == 0 and chk_start.keys_sats_mismatch.iloc[0] == 0, "starting UTXOs do not match the snapshot"
display(q(f"""SELECT grp, COUNT(*) AS n_utxo, ROUND(SUM(sats)/1e8) AS btc,
                     ROUND(SUM(IF(spent_date IS NOT NULL, sats, 0))/1e8) AS btc_spent_by_freeze,
                     COUNT(DISTINCT spending_txid) AS n_spending_tx,
                     ROUND(SUM(IF(is_patoshi, sats, 0))/1e8) AS patoshi_btc
              FROM `{DS}.q2_start_utxo` GROUP BY grp ORDER BY grp"""))

## Step 4 · Spending transactions

- `q2_tx`: transactions between 2024-12-09 and the freeze date spending starting coins.
- `q2_tx_inputs`: all inputs of these transactions (for change detection and proportional allocation), from the public `inputs` table pruned on `block_timestamp`; keys via the UDF `key_id_hex`.
- `q2_tx_outputs`: all outputs (partitions of `utxo_life` after 2024-12 only), with new-address flag, receiving-key exposure date and exchange flag.

In [ ]:
need("q2_start_utxo")
SQL_TX = f"""
CREATE OR REPLACE TABLE `{DS}.q2_tx` AS
SELECT spending_txid AS txid, ANY_VALUE(spent_date) AS spent_date
FROM `{DS}.q2_start_utxo`
WHERE spending_txid IS NOT NULL
GROUP BY spending_txid
"""
build("q2_tx", SQL_TX)

SQL_TXIN = f"""
CREATE OR REPLACE TABLE `{DS}.q2_tx_inputs` AS
WITH i AS (
  SELECT FROM_HEX(transaction_hash)   AS txid,
         DATE(block_timestamp)        AS spent_date,
         addresses[SAFE_OFFSET(0)]    AS address,
         CAST(value AS INT64)         AS sats,
         type
  FROM `{PUB}.inputs`
  WHERE block_timestamp >= TIMESTAMP('{S1}') AND block_timestamp < TIMESTAMP('{F1}')
),
j AS (
  SELECT i.* FROM i JOIN `{DS}.q2_tx` t ON t.txid = i.txid
)
SELECT j.txid, j.spent_date, j.address, j.sats, j.type,
       FROM_HEX(`{DS}.key_id_hex`(j.address)) AS key_id
FROM j
"""
build("q2_tx_inputs", SQL_TXIN)

SQL_TXOUT = f"""
CREATE OR REPLACE TABLE `{DS}.q2_tx_outputs` AS
WITH o AS (
  SELECT u.txid, u.vout, u.created_date, u.created_height, u.sats, u.type, u.address,
         IF(u.spent_date <= DATE '{F}', u.spent_date, NULL) AS out_spent_date
  FROM `{DS}.utxo_life` u
  JOIN `{DS}.q2_tx` t ON t.txid = u.txid AND t.spent_date = u.created_date
  WHERE u.created_date BETWEEN DATE '{S1}' AND DATE '{F}'
),
o2 AS (
  SELECT o.*, FROM_HEX(`{DS}.key_id_hex`(o.address)) AS dest_key_id FROM o
)
SELECT o2.*, s.first_seen_height AS addr_first_seen_height,
       e.first_exposed_date AS dest_first_exposed_date,
       ex.key_id IS NOT NULL AS to_ex
FROM o2
LEFT JOIN `{DS}.address_stats` s ON s.address = o2.address
LEFT JOIN `{DS}.key_exposure`  e ON e.key_id = o2.dest_key_id
LEFT JOIN `{DS}.exchange_keys` ex ON ex.key_id = o2.dest_key_id
"""
build("q2_tx_outputs", SQL_TXOUT)

chk_tx = q(f"""
WITH i AS (SELECT txid, SUM(sats) AS in_sats FROM `{DS}.q2_tx_inputs` GROUP BY txid),
     o AS (SELECT txid, SUM(sats) AS out_sats FROM `{DS}.q2_tx_outputs` GROUP BY txid),
     k AS (SELECT spending_txid AS txid, SUM(sats) AS k_sats FROM `{DS}.q2_start_utxo`
           WHERE spending_txid IS NOT NULL GROUP BY spending_txid)
SELECT COUNT(*) AS n_tx, COUNTIF(i.txid IS NULL) AS tx_without_inputs,
       COUNTIF(o.txid IS NULL) AS tx_without_outputs,
       COUNTIF(k.k_sats > i.in_sats) AS tx_start_gt_in,
       COUNTIF(o.out_sats > i.in_sats) AS tx_out_gt_in
FROM `{DS}.q2_tx` t
LEFT JOIN i ON i.txid = t.txid LEFT JOIN o ON o.txid = t.txid LEFT JOIN k ON k.txid = t.txid""")
display(chk_tx)
c = chk_tx.iloc[0]
assert c.tx_without_inputs == 0 and c.tx_start_gt_in == 0 and c.tx_out_gt_in == 0, "input/output checks failed"
if c.tx_without_outputs:
    print(f"Note: {c.tx_without_outputs} transactions have only outputs that are not in utxo_life, such as OP_RETURN (all counted as fee/other)")

## Step 5 · Destinations per key, `q2_key_tx` → `q2_keys`

Destination rules of 04 for non-change outputs, in order:

1. Change: receiving key is one of the transaction's input keys
2. y2 exchange: receiving key in `exchange_keys`
3. y0 exposed: P2PK, bare multisig, P2TR, or a receiving key exposed on or before the transfer day
4. y1 loose: has a key and is not exposed at the time of the transfer; y1 strict = address first seen in this transaction, hash-type, unspent for the next 30 days (transfers within 30 days of the freeze date go to `y1_strict_censored`)
5. Other: non-standard outputs without a key

Allocation: key share = input value of its starting coins / total input value (later receipts spent in the same transaction are not counted).

In [ ]:
need("q2_tx_inputs", "q2_tx_outputs", "q2_key_attr")
def _has_col(t, c):
    try: return any(f.name == c for f in client.get_table(f"{DS}.{t}").schema)
    except Exception: return False
SQL_KTX = f"""
CREATE OR REPLACE TABLE `{DS}.q2_key_tx` AS
WITH tx_in AS (
  SELECT txid, ANY_VALUE(spent_date) AS date, SUM(sats) AS in_sats, COUNT(*) AS n_in,
         ARRAY_AGG(DISTINCT key_id IGNORE NULLS) AS in_keys
  FROM `{DS}.q2_tx_inputs` GROUP BY txid
),
key_in AS (
  SELECT spending_txid AS txid, key_id, SUM(sats) AS key_in0, COUNT(*) AS n_utxo_in
  FROM `{DS}.q2_start_utxo` WHERE spending_txid IS NOT NULL
  GROUP BY spending_txid, key_id
),
o AS (
  SELECT o.txid, o.sats,
    IFNULL(o.dest_key_id IN UNNEST(t.in_keys), FALSE)                                AS is_change,
    o.to_ex,
    (o.type IN ('pubkey', 'multisig', 'witness_v1_taproot')
       OR IFNULL(o.dest_first_exposed_date <= o.created_date, FALSE))               AS exposed,
    o.dest_key_id IS NOT NULL                                                        AS keyed,
    IFNULL(o.addr_first_seen_height = o.created_height, FALSE)
      AND o.type IN {HASH_TYPES}
      AND (o.out_spent_date IS NULL
           OR o.out_spent_date > DATE_ADD(o.created_date, INTERVAL {DORMANT_DAYS} DAY)) AS strict_ok,
    DATE_ADD(o.created_date, INTERVAL {DORMANT_DAYS} DAY) > DATE '{F}'              AS censored
  FROM `{DS}.q2_tx_outputs` o
  JOIN tx_in t ON t.txid = o.txid
),
c AS (
  SELECT *,
    NOT is_change                                              AS nc,
    NOT is_change AND to_ex                                    AS c_y2,
    NOT is_change AND NOT to_ex AND exposed                    AS c_y0,
    NOT is_change AND NOT to_ex AND NOT exposed AND keyed      AS c_y1l,
    NOT is_change AND NOT to_ex AND NOT exposed AND NOT keyed  AS c_oth
  FROM o
),
tx_out AS (
  SELECT txid,
    SUM(sats)                                          AS out_sats,
    SUM(IF(is_change, sats, 0))                        AS change_sats,
    SUM(IF(nc, sats, 0))                               AS y3,
    SUM(IF(c_y2, sats, 0))                             AS y2,
    SUM(IF(c_y0, sats, 0))                             AS y0,
    SUM(IF(c_y1l, sats, 0))                            AS y1_loose,
    SUM(IF(c_y1l AND strict_ok, sats, 0))              AS y1_strict,
    SUM(IF(c_y1l AND strict_ok AND censored, sats, 0)) AS y1_strict_censored,
    SUM(IF(c_oth, sats, 0))                            AS other
  FROM c GROUP BY txid
)
SELECT ki.key_id, ki.txid, t.date, ki.key_in0, ki.n_utxo_in, t.in_sats, t.n_in,
  SAFE_DIVIDE(ki.key_in0, t.in_sats)                                          AS share,
  SAFE_DIVIDE(ki.key_in0, t.in_sats) * IFNULL(o.change_sats, 0)               AS change_sats,
  SAFE_DIVIDE(ki.key_in0, t.in_sats) * (t.in_sats - IFNULL(o.out_sats, 0))    AS fee_sats,
  SAFE_DIVIDE(ki.key_in0, t.in_sats) * IFNULL(o.y3, 0)                        AS y3,
  SAFE_DIVIDE(ki.key_in0, t.in_sats) * IFNULL(o.y2, 0)                        AS y2,
  SAFE_DIVIDE(ki.key_in0, t.in_sats) * IFNULL(o.y0, 0)                        AS y0,
  SAFE_DIVIDE(ki.key_in0, t.in_sats) * IFNULL(o.y1_loose, 0)                  AS y1_loose,
  SAFE_DIVIDE(ki.key_in0, t.in_sats) * IFNULL(o.y1_strict, 0)                 AS y1_strict,
  SAFE_DIVIDE(ki.key_in0, t.in_sats) * IFNULL(o.y1_strict_censored, 0)        AS y1_strict_censored,
  SAFE_DIVIDE(ki.key_in0, t.in_sats) * IFNULL(o.other, 0)                     AS other
FROM key_in ki
JOIN tx_in t ON t.txid = ki.txid
LEFT JOIN tx_out o ON o.txid = ki.txid
"""
build("q2_key_tx", SQL_KTX)

SQL_KEYS = f"""
CREATE OR REPLACE TABLE `{DS}.q2_keys` AS
WITH su AS (
  SELECT key_id,
         SUM(sats)                                   AS s0_chk,
         SUM(IF(spent_date IS NULL, sats, 0))        AS remain0,
         SUM(IF(spent_date IS NOT NULL, sats, 0))    AS spent0,
         MIN(spent_date)                             AS first_move_date,
         LOGICAL_OR(is_patoshi)                      AS is_patoshi,
         SUM(IF(is_patoshi, sats, 0))                AS patoshi_sats,
         LOGICAL_AND(type = 'pubkey' AND created_date <= DATE '{EARLY_END}') AS all_early_p2pk
  FROM `{DS}.q2_start_utxo` GROUP BY key_id
),
mj AS (   -- last "major" receipt (a single receipt >= MAJOR_IN_SHARE of the starting balance), so that dust does not "refresh" old never-spent keys
  SELECT s.key_id, MAX(s.created_date) AS newest_major_in_date
  FROM `{DS}.q2_start_utxo` s JOIN `{DS}.q2_universe` u ON u.key_id = s.key_id
  WHERE s.sats >= {MAJOR_IN_SHARE} * u.s0_sats
  GROUP BY s.key_id
),
f AS (
  SELECT key_id, COUNT(*) AS n_move_tx, SUM(key_in0) AS f_in0,
         SUM(change_sats) AS change_sats, SUM(fee_sats) AS fee_sats, SUM(y3) AS y3, SUM(y2) AS y2,
         SUM(y0) AS y0, SUM(y1_loose) AS y1_loose, SUM(y1_strict) AS y1_strict,
         SUM(y1_strict_censored) AS y1_strict_censored, SUM(other) AS other
  FROM `{DS}.q2_key_tx` GROUP BY key_id
)
SELECT u.key_id, u.grp, u.s0_sats, u.s0_n_utxo, u.oldest_utxo_date, u.newest_utxo_date,
       u.first_exposed_date, u.first_exposed_src, mj.newest_major_in_date,
       a.first_seen_date, a.last_spend_date, a.last_received_date, a.max_addr_n_received,
       IFNULL(a.end_sats, 0) AS end_sats, IFNULL(a.is_exchange, FALSE) AS is_exchange,
       IFNULL(a.is_custodial, FALSE) AS is_custodial, IFNULL(a.is_service, FALSE) AS is_service,
       su.s0_chk, su.remain0, su.spent0, su.first_move_date, su.is_patoshi, su.patoshi_sats, su.all_early_p2pk,
       IFNULL(f.n_move_tx, 0) AS n_move_tx, IFNULL(f.f_in0, 0) AS f_in0,
       IFNULL(f.change_sats, 0) AS change_sats, IFNULL(f.fee_sats, 0) AS fee_sats,
       IFNULL(f.y3, 0) AS y3, IFNULL(f.y2, 0) AS y2, IFNULL(f.y0, 0) AS y0,
       IFNULL(f.y1_loose, 0) AS y1_loose, IFNULL(f.y1_strict, 0) AS y1_strict,
       IFNULL(f.y1_strict_censored, 0) AS y1_strict_censored, IFNULL(f.other, 0) AS other
FROM `{DS}.q2_universe` u
LEFT JOIN `{DS}.q2_key_attr` a ON a.key_id = u.key_id
LEFT JOIN su ON su.key_id = u.key_id
LEFT JOIN f  ON f.key_id  = u.key_id
LEFT JOIN mj ON mj.key_id = u.key_id
"""
build("q2_keys", SQL_KEYS, force=not _has_col("q2_keys", "newest_major_in_date"))   # rebuild if newest_major_in_date is missing

### Checks

1. Identities: starting coins = unmoved + spent; spent = change + fee + y3; y3 = y2 + y0 + y1 loose + other; y1 strict ≤ y1 loose.
2. Shares of starting keys in one transaction sum to ≤ 1.
3. Balance at the freeze date ≥ unmoved part of the starting coins (unmoved coins are still held by the key).
4. Comparison with the known figure from 02: group L starts with about 6.2 million BTC.

In [ ]:
need("q2_keys")
chk = q(f"""
SELECT COUNT(*) AS n_keys,
  COUNTIF(s0_chk IS NULL OR s0_chk != s0_sats)                 AS bad_s0,
  COUNTIF(remain0 + spent0 != s0_sats)                         AS bad_remain_spent,
  COUNTIF(ABS(f_in0 - spent0) > 1)                             AS bad_flow_total,
  COUNTIF(ABS(change_sats + fee_sats + y3 - f_in0) > 1 + 1e-6 * f_in0) AS bad_spent_identity,
  COUNTIF(ABS(y2 + y0 + y1_loose + other - y3) > 1 + 1e-6 * y3) AS bad_y3_identity,
  COUNTIF(y1_strict > y1_loose + 1)                            AS bad_strict,
  COUNTIF(fee_sats < -1)                                       AS neg_fee,
  COUNTIF(end_sats < remain0)                                  AS end_lt_remain,
  COUNTIF(first_seen_date IS NULL)                             AS no_attr,
  ROUND(SUM(IF(grp = 'L', s0_sats, 0))/1e8) AS L_btc, ROUND(SUM(IF(grp = 'T', s0_sats, 0))/1e8) AS T_btc
FROM `{DS}.q2_keys`""")
display(chk)
share = q(f"""SELECT COUNTIF(s > 1 + 1e-9) AS tx_share_gt_1, MAX(s) AS max_share_sum
              FROM (SELECT txid, SUM(share) AS s FROM `{DS}.q2_key_tx` GROUP BY txid)""")
display(share)
c = chk.iloc[0]
for k in ["bad_s0", "bad_remain_spent", "bad_flow_total", "bad_spent_identity", "bad_y3_identity",
          "bad_strict", "neg_fee", "end_lt_remain"]:
    assert c[k] == 0, f"check failed: {k} = {c[k]}"
assert share.tx_share_gt_1.iloc[0] == 0
print("Identity checks passed. Group L starts with %.2fM BTC (pre-Willow figure from 02: about 6.20M)" % (c.L_btc / 1e6))

## Step 6 · Cells, `q2_cells` (pandas from here on)

Keys aggregated to cells (key counts and BTC totals); the segment table, sensitivity and heterogeneity are computed from this table:

- `grp` (L/T), `is_service`, `moved` (starting coins spent), `dest_cls` (main destination of movers: `y1s` if y1 strict is more than half of the outflow; otherwise the largest of y1 loose-not-strict / y0 / exchange / other; `self` if change only);
- `end_ge` (balance at the freeze date ≥ 0.01 BTC), `spent_since` (any spend after 2024-12-09);
- `ref_years`: whole years before the freeze date of the last spend (never-spent keys: receipt date of the newest starting UTXO), `never_spent`;
- `bal_bin` (starting balance < 0.01 / 0.01–1 / 1–10 / 10–100 / ≥ 100), `fs_year` (first-seen year), `is_patoshi`, `all_early_p2pk`.

In [ ]:
need("q2_keys")
SQL_CELLS = f"""
WITH k AS (
  SELECT *,
    spent0 > 0 AS moved,
    end_sats >= {SATS_MIN} AS end_ge,
    IFNULL(last_spend_date >= DATE '{S1}', FALSE) AS spent_since,
    last_spend_date IS NULL AS never_spent,
    COALESCE(LEAST(last_spend_date, DATE '{F}'), newest_major_in_date, newest_utxo_date) AS ref_date,
    COALESCE(newest_major_in_date, newest_utxo_date) < newest_utxo_date AS dust_after_major,
    y1_loose - y1_strict AS y1_ns
  FROM `{DS}.q2_keys`
)
SELECT grp, is_service, moved,
  CASE WHEN NOT moved THEN 'none'
       WHEN y3 <= 0 THEN 'self'
       WHEN y1_strict >= {Y1S_MAJORITY} * y3 THEN 'y1s'
       WHEN y1_ns >= GREATEST(y0, y2, other) THEN 'y1_loose_ns'
       WHEN y0 >= GREATEST(y2, other) THEN 'y0'
       WHEN y2 >= other THEN 'y2'
       ELSE 'other' END AS dest_cls,
  end_ge, spent_since, never_spent,
  CAST(FLOOR(DATE_DIFF(DATE '{F}', ref_date, DAY) / 365.25) AS INT64) AS ref_years,
  CASE WHEN s0_sats < {SATS_MIN} THEN '0 <0.01' WHEN s0_sats < 1e8 THEN '1 0.01-1'
       WHEN s0_sats < 1e9 THEN '2 1-10' WHEN s0_sats < 1e10 THEN '3 10-100' ELSE '4 >=100' END AS bal_bin,
  EXTRACT(YEAR FROM first_seen_date) AS fs_year,
  IFNULL(is_patoshi, FALSE) AS is_patoshi, IFNULL(all_early_p2pk, FALSE) AS all_early_p2pk,
  dust_after_major,
  COUNT(*) AS n_keys,
  SUM(s0_sats)/1e8 AS s0_btc, SUM(end_sats)/1e8 AS end_btc, SUM(remain0)/1e8 AS remain0_btc,
  SUM(spent0)/1e8 AS spent0_btc, SUM(y1_strict)/1e8 AS y1s_btc, SUM(y1_strict_censored)/1e8 AS y1s_cens_btc,
  SUM(y1_ns)/1e8 AS y1ns_btc, SUM(y0)/1e8 AS y0_btc, SUM(y2)/1e8 AS y2_btc, SUM(other)/1e8 AS other_btc,
  SUM(change_sats)/1e8 AS change_btc, SUM(fee_sats)/1e8 AS fee_btc, SUM(patoshi_sats)/1e8 AS patoshi_btc
FROM k
GROUP BY grp, is_service, moved, dest_cls, end_ge, spent_since, never_spent, ref_years, bal_bin,
         fs_year, is_patoshi, all_early_p2pk, dust_after_major
"""
gb, cost = dry_run(SQL_CELLS); print(f"q2_cells estimated {gb:,.1f} GB ~ ${cost:.2f}")
cells = q(SQL_CELLS)
for col in ["is_service", "moved", "end_ge", "spent_since", "never_spent", "is_patoshi", "all_early_p2pk", "dust_after_major"]:
    cells[col] = cells[col].astype(bool)
cells["fs_year"] = cells.fs_year.astype("Int64")
cells.to_csv("q2_cells.csv.gz", index=False)
print(f"{len(cells):,} cells, {cells.n_keys.sum():,} keys, {cells.s0_btc.sum():,.0f} BTC")
_d = cells[cells.never_spent & cells.dust_after_major & cells.end_ge]
print(f"Never-spent keys whose last receipt is dust (< 1% of the starting balance): {_d.n_keys.sum():,} keys, {_d.s0_btc.sum():,.0f} BTC"
      f" (dormancy measured from the last receipt would let dust 'refresh' these keys; the last major receipt is used instead)")

## Step 7 · Segment table, key number, sensitivity, heterogeneity

- Key number: keys that spent at least once after 2024-12-09 and still hold ≥ 0.01 BTC exposed at the freeze date (subset of C), with their BTC.
- Sensitivity: activity window 1/3/5 years × dormancy threshold 5/10/15 years.
- Heterogeneity: balance and first-seen year. Groups L and T.

In [ ]:
SEG_ORDER = ["A Confirmed migrated", "B Moved, still exposed", "C Active, not migrated", "D Indeterminate",
             "E Behaviorally unreachable (lower bound)", "Dust <0.01 BTC", "Exchange/custodian"]
SEG_SHORT = {"A Confirmed migrated": "Confirmed migrated", "B Moved, still exposed": "Moved, still exposed",
          "C Active, not migrated": "Active, not migrated", "D Indeterminate": "Indeterminate",
          "E Behaviorally unreachable (lower bound)": "Behaviourally unreachable", "Dust <0.01 BTC": "Dust <0.01 BTC",
          "Exchange/custodian": "Exchange / custodian"}

def segment(df, active=ACTIVE_YEARS, dormant=DORMANT_YEARS):
    """Assign each cell to one class (checked in order, mutually exclusive)."""
    assert active <= dormant
    seg = np.select(
        [df.is_service,
         df.moved & ~df.end_ge & (df.dest_cls == "y1s"),
         df.moved & ~df.end_ge,
         df.end_ge & ~df.never_spent & (df.ref_years < active),
         df.end_ge & (df.ref_years >= dormant),
         df.end_ge,
         ~df.end_ge & ~df.moved],
        SEG_ORDER[6:7] + SEG_ORDER[0:3] + [SEG_ORDER[4], SEG_ORDER[3], SEG_ORDER[5]],
        default="?")
    assert (seg != "?").all()
    return pd.Series(seg, index=df.index)

def seg_table(df, active=ACTIVE_YEARS, dormant=DORMANT_YEARS):
    d = df.assign(seg=segment(df, active, dormant))
    t = d.groupby(["grp", "seg"]).agg(n_keys=("n_keys", "sum"), s0_btc=("s0_btc", "sum"),
                                      end_btc=("end_btc", "sum"), patoshi_btc=("patoshi_btc", "sum")).reset_index()
    t["share_btc"] = t.s0_btc / t.groupby("grp").s0_btc.transform("sum")
    t["share_keys"] = t.n_keys / t.groupby("grp").n_keys.transform("sum")
    t["seg"] = pd.Categorical(t.seg, SEG_ORDER, ordered=True)
    return t.sort_values(["grp", "seg"]).reset_index(drop=True)

seg = seg_table(cells)
seg["segment_en"] = seg.seg.map(SEG_SHORT)
seg.to_csv("q2_segments.csv", index=False)
fmt = {"n_keys": "{:,.0f}", "s0_btc": "{:,.0f}", "end_btc": "{:,.0f}", "patoshi_btc": "{:,.0f}",
       "share_btc": "{:.1%}", "share_keys": "{:.1%}"}
for g in ["L", "T"]:
    print(f"== group {g} (start {START.date()}, followed to {FREEZE.date()}; main definition: active {ACTIVE_YEARS} years / dormant {DORMANT_YEARS} years)")
    display(seg[seg.grp == g].drop(columns="grp").style.format(fmt).hide(axis="index"))

# class B split by main destination
b = cells[~cells.is_service & cells.moved & ~cells.end_ge & (cells.dest_cls != "y1s")]
b_split = b.groupby(["grp", "dest_cls"])[["n_keys", "s0_btc"]].sum().reset_index()
print("Class B by main destination (y1_loose_ns = to a new address but moved again within 30 days, unclear; self = change only, not followed further)")
display(b_split)

# key number: spent after 2024-12-09 and still holds >= 0.01 BTC exposed at the freeze date (non-service)
kn = (cells[~cells.is_service & cells.spent_since & cells.end_ge]
      .groupby("grp").agg(n_keys=("n_keys", "sum"), end_btc=("end_btc", "sum"), s0_btc=("s0_btc", "sum"),
                          moved_start_coins_keys=("n_keys", lambda s: s[cells.loc[s.index, "moved"]].sum()))
      .reset_index())
kn.to_csv("q2_key_number.csv", index=False)
print("Key number (keys that spent at least once after Willow and still hold >= 0.01 BTC exposed at the freeze date; end_btc = exposed balance at the freeze date)")
display(kn)

# Patoshi and early mining
pat = cells.assign(seg=segment(cells)).query("is_patoshi or all_early_p2pk")
display(pat.groupby(["is_patoshi", "all_early_p2pk", "seg"])[["n_keys", "s0_btc", "patoshi_btc"]].sum()
           .query("n_keys > 0"))

In [ ]:
# sensitivity: activity window x dormancy threshold
rows = []
for a in SENS_ACTIVE:
    for d in SENS_DORMANT:
        if a > d: continue
        t = seg_table(cells, a, d)
        for r in t.itertuples():
            rows.append({"active_years": a, "dormant_years": d, "grp": r.grp, "seg": str(r.seg),
                         "n_keys": r.n_keys, "s0_btc": r.s0_btc, "share_btc": r.share_btc})
sens = pd.DataFrame(rows)
sens.to_csv("q2_sensitivity.csv", index=False)
piv = sens[(sens.grp == "L") & sens.seg.isin(SEG_ORDER[2:5])].pivot_table(
    index=["active_years", "dormant_years"], columns="seg", values="s0_btc").reindex(columns=SEG_ORDER[2:5])
print("BTC (10,000 BTC) of classes C/D/E of group L by threshold; A, B, dust and exchanges do not depend on these two thresholds")
display((piv / 1e4).round(1))

# heterogeneity: balance and first-seen year (main definition)
cm = cells.assign(seg=segment(cells))
def fs_bin(y):
    if pd.isna(y): return "unknown"
    y = int(y)
    return "≤2010" if y <= 2010 else "2011–15" if y <= 2015 else "2016–19" if y <= 2019 else "2020–22" if y <= 2022 else "≥2023"
cm["fs_bin"] = cm.fs_year.map(fs_bin)
het_bal = cm.groupby(["grp", "bal_bin", "seg"])[["n_keys", "s0_btc"]].sum().reset_index()
het_bal["share_btc"] = het_bal.s0_btc / het_bal.groupby(["grp", "bal_bin"]).s0_btc.transform("sum")
het_fsy = cm.groupby(["grp", "fs_bin", "seg"])[["n_keys", "s0_btc"]].sum().reset_index()
het_fsy["share_btc"] = het_fsy.s0_btc / het_fsy.groupby(["grp", "fs_bin"]).s0_btc.transform("sum")
het_bal.to_csv("q2_hetero_bal.csv", index=False); het_fsy.to_csv("q2_hetero_fsy.csv", index=False)
print("Group L by starting balance (BTC shares)")
display(het_bal[het_bal.grp == "L"].pivot_table(index="bal_bin", columns="seg", values="share_btc")
        .reindex(columns=SEG_ORDER).style.format("{:.1%}", na_rep="–"))
print("Group L by first-seen year (BTC shares)")
display(het_fsy[het_fsy.grp == "L"].pivot_table(index="fs_bin", columns="seg", values="share_btc")
        .reindex(index=["≤2010", "2011–15", "2016–19", "2020–22", "≥2023", "unknown"], columns=SEG_ORDER)
        .dropna(how="all").style.format("{:.1%}", na_rep="–"))

## Step 8 · Coin-level decomposition and stock bridge

- Coin level: starting coins = unmoved at the freeze date + spent; spent coins split into y1 strict (cases not observable for 30 days listed separately) / y1 loose-not-strict / y0 / exchange / other / change / fee; for non-service keys and for all keys.
- Stock bridge: group-L stock at the freeze date = unmoved starting coins + coins later received by starting keys (including change) + keys in group C at Willow exposed later + keys not in the Willow snapshot.

In [ ]:
def coin_decomp(df):
    s = df[["s0_btc", "remain0_btc", "spent0_btc", "y1s_btc", "y1s_cens_btc", "y1ns_btc", "y0_btc",
            "y2_btc", "other_btc", "change_btc", "fee_btc"]].sum()
    return s
dec = pd.concat({(g, lab): coin_decomp(cells[(cells.grp == g) & m])
                 for g in ["L", "T"]
                 for lab, m in [("non_service", ~cells.is_service), ("all", cells.is_service | ~cells.is_service)]},
                axis=1).T
dec.index.names = ["grp", "keys"]
dec.to_csv("q2_coin_decomp.csv")
print("Where the starting coins went (BTC): y1s_cens = y1 strict received within 30 days before the freeze date, not observable for 30 days")
display(dec.round(0))

SQL_BRIDGE = f"""
WITH w AS (SELECT key_id, grp FROM `{DS}.holder_snapshot` WHERE event = '{START_EVENT}'),
l AS (SELECT key_id, sats FROM `{DS}.holder_snapshot` WHERE event = 'latest' AND grp = 'L')
SELECT CASE WHEN w.key_id IS NULL THEN 'not_in_start_snapshot'
            WHEN w.grp = 'C' THEN 'was_C_at_start' ELSE CONCAT('was_', w.grp, '_at_start') END AS src,
       COUNT(*) AS n_keys, SUM(l.sats)/1e8 AS btc
FROM l LEFT JOIN w ON w.key_id = l.key_id
GROUP BY src
"""
br = q(SQL_BRIDGE).set_index("src")
L = cells[cells.grp == "L"]
bridge = pd.DataFrame([
    ("Starting exposed supply (pre-Willow, group L)", L.s0_btc.sum()),
    ("- Starting coins spent", -L.spent0_btc.sum()),
    ("= Starting coins unmoved at freeze date", L.remain0_btc.sum()),
    ("+ Later receipts by starting keys (incl. change)", L.end_btc.sum() - L.remain0_btc.sum()),
    ("+ Keys in group C at Willow, exposed later", br.btc.get("was_C_at_start", 0.0)),
    ("+ Keys not in the Willow snapshot", br.btc.get("not_in_start_snapshot", 0.0)),
    ("= Group L stock at freeze date (bridge total)", L.remain0_btc.sum() + (L.end_btc.sum() - L.remain0_btc.sum())
       + br.btc.get("was_C_at_start", 0.0) + br.btc.get("not_in_start_snapshot", 0.0)),
], columns=["item", "btc"])
latest_L = q(f"SELECT SUM(sats)/1e8 AS btc FROM `{DS}.holder_snapshot` WHERE event = 'latest' AND grp = 'L'").btc.iloc[0]
gap = bridge.btc.iloc[-1] - latest_L
print(f"Bridge total {bridge.btc.iloc[-1]:,.0f} vs group L in the latest snapshot {latest_L:,.0f}, difference {gap:+,.1f} BTC")
assert abs(gap) < 1, "bridge does not add up"
assert abs(br.btc.get("was_L_at_start", 0.0) - L.end_btc.sum()) < 1, "freeze-date balance of starting L keys does not match"
bridge.to_csv("q2_bridge.csv", index=False)
display(bridge.round(0))
display(br)

## Step 9 · Figures
- `fig_30_segments.png`: segments of groups L and T (share of starting BTC, stacked horizontal bars, labeled directly).
- `fig_30_sensitivity.png`: BTC of classes C and E of group L by threshold.
- `fig_30_bridge.png`: bridge of the group-L stock from before Willow to the freeze date (waterfall).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
PAL = dict(zip(SEG_ORDER, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#b8b7b1", "#8a8983"]))
INK, INK2 = "#0b0b0b", "#52514e"

fig, ax = plt.subplots(figsize=(11, 2.9))
for yi, g in enumerate(["L", "T"]):
    t = seg[seg.grp == g]
    left = 0.0
    tot = t.s0_btc.sum()
    for r in t.itertuples():
        w = r.share_btc
        ax.barh(yi, w, left=left, color=PAL[str(r.seg)], edgecolor="white", linewidth=2, height=0.62)
        if w >= 0.045:
            ax.text(left + w / 2, yi, f"{w:.0%}\n{r.s0_btc/1e6:.2f}M" if tot > 1e6 else f"{w:.0%}\n{r.s0_btc/1e3:.0f}k",
                    ha="center", va="center", fontsize=8, color=INK)
        left += w
ax.set_yticks([0, 1], [f"Legacy exposed (L)\n{seg[seg.grp=='L'].s0_btc.sum()/1e6:.2f}M BTC",
                       f"Taproot (T)\n{seg[seg.grp=='T'].s0_btc.sum()/1e3:.0f}k BTC"], color=INK)
ax.invert_yaxis(); ax.set_xlim(0, 1); ax.set_xlabel("share of exposed supply on 2024-12-08 (BTC)", color=INK2)
for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
ax.legend(handles=[Patch(color=PAL[s], label=SEG_SHORT[s]) for s in SEG_ORDER], ncol=4, fontsize=8,
          frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.32))
ax.set_title(f"Where the exposed supply of {START.date()} stood on {FREEZE.date()} "
             f"(active ≤{ACTIVE_YEARS}y, unreachable ≥{DORMANT_YEARS}y)", fontsize=10, loc="left", color=INK)
plt.savefig("fig_30_segments.png", dpi=150, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, s in zip(axes, [SEG_ORDER[2], SEG_ORDER[4]]):
    m = sens[(sens.grp == "L") & (sens.seg == s)].pivot_table(index="active_years", columns="dormant_years", values="s0_btc")
    for a, colr in zip(m.index, ["#2a78d6", "#eb6834", "#1baf7a"]):
        ax.plot(m.columns, m.loc[a] / 1e6, marker="o", ms=8, lw=2, color=colr, label=f"active ≤{a}y")
    ax.set_title(SEG_SHORT[s], fontsize=10, loc="left", color=INK); ax.set_xlabel("dormancy threshold (years)", color=INK2)
    ax.set_xticks(SENS_DORMANT)
    for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
    ax.grid(axis="y", color="#e6e5e0", lw=0.8)
axes[0].set_ylabel("BTC (millions), legacy exposed", color=INK2); axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig("fig_30_sensitivity.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(10, 3.6))
steps = [("Start (L)", L.s0_btc.sum(), "total"), ("Start coins spent", -L.spent0_btc.sum(), "delta"),
         ("New coins to\nstart keys", L.end_btc.sum() - L.remain0_btc.sum(), "delta"),
         ("Keys newly\nexposed (were C)", br.btc.get("was_C_at_start", 0.0), "delta"),
         ("Keys not in\nstart snapshot", br.btc.get("not_in_start_snapshot", 0.0), "delta"),
         ("Freeze (L)", latest_L, "total")]
run = 0.0
off = 0.015 * max(L.s0_btc.sum(), latest_L) / 1e6
for i, (lab, v, kind) in enumerate(steps):
    if kind == "total":
        ax.bar(i, v / 1e6, color="#52514e", width=0.6); run = v; y = v
    else:
        ax.bar(i, v / 1e6, bottom=run / 1e6, color="#2a78d6" if v >= 0 else "#e34948", width=0.6); run += v; y = max(run, run - v)
    ax.text(i, y / 1e6 + off, f"{v/1e6:+.2f}M" if kind == "delta" else f"{v/1e6:.2f}M", ha="center", fontsize=8, color=INK)
ax.set_xticks(range(len(steps)), [s[0] for s in steps], fontsize=8, color=INK)
ax.set_ylabel("BTC (millions)", color=INK2)
for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
ax.set_title("Legacy exposed stock: 2024-12-08 → 2026-09-22", fontsize=10, loc="left", color=INK)
plt.tight_layout(); plt.savefig("fig_30_bridge.png", dpi=150); plt.show()

In [ ]:
for rec in JOBS[-6:]:
    job = client.get_job(rec["job_id"], location=rec.get("location", LOCATION))
    print(rec["table"], job.state, "errors:", job.errors)
    if job.state != "DONE":
        job.result(); print("  done")
TABLES_30 = ["patoshi_blocks", "q2_universe", "q2_key_attr", "q2_start_utxo", "q2_tx", "q2_tx_inputs",
             "q2_tx_outputs", "q2_key_tx", "q2_keys"]
for t in TABLES_30:
    n = table_rows(t)
    print(f"{t:16s}", f"{n:,} rows" if n else "not built yet")

In [ ]:
# step 1 skipped: Patoshi flags from the table
if "patoshi" not in globals():
    patoshi = q(f"SELECT height FROM `{DS}.patoshi_blocks` WHERE height >= 0")
if "PATOSHI_SRC" not in globals():
    PATOSHI_SRC = PATOSHI_URL if len(patoshi) else None
TABLES_30 = ["patoshi_blocks", "q2_universe", "q2_key_attr", "q2_start_utxo", "q2_tx", "q2_tx_inputs",
             "q2_tx_outputs", "q2_key_tx", "q2_keys"]
report = {"generated_at": dt.datetime.now(dt.timezone.utc).isoformat(),
          "config": {"start_event": START_EVENT, "start_snap": S, "freeze": F, "min_btc": MIN_BTC,
                     "service_max_recv": SERVICE_MAX_RECV, "dormant_days": DORMANT_DAYS,
                     "y1s_majority": Y1S_MAJORITY, "active_years": ACTIVE_YEARS, "dormant_years": DORMANT_YEARS,
                     "patoshi_source": PATOSHI_SRC, "n_patoshi_blocks": int(len(patoshi))},
          "ledger": LEDGER,
          "tables": [table_info(t) for t in TABLES_30 if table_rows(t)]}
for k, v in [("check_start", "chk_start"), ("check_tx", "chk_tx"), ("checks", "chk"),
             ("segments", "seg"), ("key_number", "kn"), ("bridge", "bridge")]:
    try: report[k] = eval(v).astype(str).to_dict(orient="records")
    except NameError: pass
with open("build_report_30.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
OUT_30 = ["build_report_30.json", "q2_segments.csv", "q2_key_number.csv", "q2_sensitivity.csv",
          "q2_hetero_bal.csv", "q2_hetero_fsy.csv", "q2_coin_decomp.csv", "q2_bridge.csv", "q2_cells.csv.gz",
          "fig_30_segments.png", "fig_30_sensitivity.png", "fig_30_bridge.png"]
try:
    from google.colab import files
    for fn in OUT_30:
        if os.path.exists(fn): files.download(fn)
except Exception:
    print("All files are in the working directory")